# CardioIA - Fase 4
## Parte 1: Pre-processamento de Imagens Medicas

| Campo | Info |
|---|---|
| **Projeto** | CardioIA - Assistente Cardiologico Virtual |
| **Dataset** | PneumoniaMNIST (MedMNIST) — download automatico, sem autenticacao |
| **Objetivo** | Pre-processar imagens de raio-X de torax para classificacao por CNN |

---

## 1. Instalacao e Configuracao

In [ ]:
!pip install medmnist -q

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from medmnist import PneumoniaMNIST

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

## 2. Download do Dataset

**PneumoniaMNIST** (MedMNIST v2)
- Derivado do dataset Chest X-Ray (Pneumonia) do Kaggle
- ~5.800 imagens de raio-X de torax
- Classes: **NORMAL (0)** e **PNEUMONIA (1)**
- Download automatico sem necessidade de autenticacao

In [ ]:
IMG_SIZE   = 64   # compacto: economiza RAM e acelera treinamento
BATCH_SIZE = 32
CLASSES    = ["NORMAL", "PNEUMONIA"]

print("Baixando dataset...")
raw_train = PneumoniaMNIST(split='train', download=True, size=IMG_SIZE)
raw_val   = PneumoniaMNIST(split='val',   download=True, size=IMG_SIZE)
raw_test  = PneumoniaMNIST(split='test',  download=True, size=IMG_SIZE)

print(f"Treino:    {len(raw_train)} imagens")
print(f"Validacao: {len(raw_val)} imagens")
print(f"Teste:     {len(raw_test)} imagens")
print("Download concluido!")

## 3. Analise Exploratoria

In [ ]:
# Distribuicao por classe
splits_info = {
    "Treino":    raw_train,
    "Validacao": raw_val,
    "Teste":     raw_test
}

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(3); w = 0.35

for idx, (split_name, ds) in enumerate(splits_info.items()):
    labels = np.array([int(label[0]) for _, label in ds])
    n_normal    = (labels == 0).sum()
    n_pneumonia = (labels == 1).sum()
    if idx == 0:
        b1 = ax.bar(idx - w/2, n_normal,    w, color="#27AE60", alpha=0.85, label="NORMAL")
        b2 = ax.bar(idx + w/2, n_pneumonia, w, color="#E74C3C", alpha=0.85, label="PNEUMONIA")
    else:
        ax.bar(idx - w/2, n_normal,    w, color="#27AE60", alpha=0.85)
        ax.bar(idx + w/2, n_pneumonia, w, color="#E74C3C", alpha=0.85)

ax.set_xticks(x); ax.set_xticklabels(["Treino", "Validacao", "Teste"])
ax.set_ylabel("Qtd. de Imagens")
ax.set_title("Distribuicao do Dataset por Classe e Split")
ax.legend(); ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig("distribuicao_dataset.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: distribuicao_dataset.png")

In [ ]:
# Amostras originais (antes do pre-processamento)
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.suptitle("Amostras Originais - Raios-X de Torax", fontsize=14, fontweight="bold")

for i, cls_idx in enumerate([0, 1]):
    cls_name = CLASSES[cls_idx]
    amostras = [(img, lbl) for img, lbl in raw_train if int(lbl[0]) == cls_idx][:4]
    for j, (img, _) in enumerate(amostras):
        axes[i][j].imshow(img, cmap="gray")
        color = "#27AE60" if cls_idx == 0 else "#E74C3C"
        axes[i][j].set_title(cls_name, color=color, fontsize=10, fontweight="bold")
        axes[i][j].axis("off")

plt.tight_layout()
plt.savefig("amostras_originais.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: amostras_originais.png")

## 4. Pre-processamento

| Tecnica | Detalhes |
|---|---|
| Redimensionamento | 64x64 px (eficiente para CNN; VGG16 faz resize interno) |
| Normalizacao | Pixels [0-255] -> [0-1] (divisao por 255) |
| Formato de cor | RGB (3 canais) |
| Data Augmentation | Rotacao +-10 graus, flip horizontal, shift H/V 10% (so treino) |

In [ ]:
def to_arrays(dataset):
    X = np.stack([
        np.array(img.convert("RGB")).astype(np.float32) / 255.0
        for img, _ in dataset
    ])
    y = np.array([int(lbl[0]) for _, lbl in dataset])
    return X, y

print("Convertendo imagens para arrays normalizados...")
X_train, y_train = to_arrays(raw_train)
X_val,   y_val   = to_arrays(raw_val)
X_test,  y_test  = to_arrays(raw_test)

print(f"Shape treino:    {X_train.shape}")
print(f"Shape validacao: {X_val.shape}")
print(f"Shape teste:     {X_test.shape}")
print(f"Valores: min={X_train.min():.2f}  max={X_train.max():.2f}")
print("Normalizacao aplicada!")

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Augmentation para treino
aug = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode="nearest"
)

# Generator de treino com augmentation
train_gen = aug.flow(X_train, y_train, batch_size=BATCH_SIZE)

# Visualizar imagens com augmentation aplicado
batch_imgs, batch_labels = next(train_gen)

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.suptitle("Imagens Pre-processadas (64x64, normalizadas, com augmentation)", fontsize=12, fontweight="bold")
for i in range(8):
    ax = axes[i // 4][i % 4]
    ax.imshow(batch_imgs[i])
    label = CLASSES[int(batch_labels[i])]
    color = "#27AE60" if batch_labels[i] == 0 else "#E74C3C"
    ax.set_title(label, color=color, fontsize=10, fontweight="bold")
    ax.set_xlabel(f"min={batch_imgs[i].min():.2f}  max={batch_imgs[i].max():.2f}", fontsize=7)
    ax.axis("off")

plt.tight_layout()
plt.savefig("amostras_preprocessadas.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: amostras_preprocessadas.png")

## 5. Resumo do Pipeline

In [ ]:
print("=" * 55)
print("   RESUMO - PIPELINE DE PRE-PROCESSAMENTO")
print("=" * 55)
print(f"  Dataset:        PneumoniaMNIST (MedMNIST v2)")
print(f"  Fonte original: Chest X-Ray Pneumonia (Kaggle)")
print(f"  Tarefa:         Classificacao binaria")
print(f"  Classes:        NORMAL (0) / PNEUMONIA (1)")
print(f"  Resolucao:      {IMG_SIZE} x {IMG_SIZE} pixels")
print(f"  Normalizacao:   [0, 1]  (divisao por 255)")
print(f"  Canais:         RGB (3)")
print(f"  Batch size:     {BATCH_SIZE}")
print(f"  Treino:         {X_train.shape[0]} imgs")
print(f"  Validacao:      {X_val.shape[0]} imgs")
print(f"  Teste:          {X_test.shape[0]} imgs")
print("-" * 55)
print("  Augmentation (treino):")
print("    Rotacao +- 10 graus")
print("    Shift horizontal e vertical: 10%")
print("    Flip horizontal")
print("=" * 55)
print("\nPre-processamento concluido!")

## Conclusao

Pipeline de pre-processamento concluido:

- Dataset PneumoniaMNIST: ~5.800 imagens de raio-X divididas em treino, validacao e teste
- Imagens padronizadas: 64x64 px, RGB, normalizadas em [0, 1]
- Data Augmentation aplicado ao treino para melhorar generalizacao
- Arrays numpy prontos para alimentar os modelos CNN (Parte 2)

Imagens geradas: distribuicao_dataset.png, amostras_originais.png, amostras_preprocessadas.png